# LangChain L12 — Level 11 — LangGraph workflows and persistence
Some processes should not be left to a model's judgement at every step. A support ticket at
Meridian must follow a fixed shape: classify, gather evidence in the right system, draft, get
human approval, send. That is a **workflow**, and `create_agent()`'s free-form loop is the
wrong tool for it. LangGraph is the layer underneath: you declare the graph yourself.

```text
START -> classify -+-> faq (policy search) ---+-> draft -> approve (interrupt) -> send -> END
                   +-> billing (order lookup) +
```

- **State** is a typed dictionary that flows through the graph.
- **Nodes** are Python functions that read state and return updates.
- **Edges** connect nodes; **conditional edges** choose the next node from state.
- A **checkpointer** makes the graph resumable; `interrupt()` pauses it inside a node.

`create_agent()` is itself a LangGraph graph with a *model* node and a *tools* node. Once you
can build this section's graph, you can build any agent shape, and you can mix both: an agent
can be one node of a larger workflow.

### Step 1 — State and nodes

Each node does one job and returns only the keys it changes. Model calls appear where they add
value (classification, drafting); deterministic work (order lookup) is plain Python.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt

class RouteDecision(BaseModel):
    """Which desk should handle the ticket."""
    category: Literal["faq", "billing"] = Field(description="billing for orders, charges and refunds; faq for policy questions.")

def structured(model, schema):
    """model.with_structured_output for the course model (function calling is the most portable method on OpenRouter)."""
    return model.with_structured_output(schema, method="function_calling") if LIVE else model.with_structured_output(schema)

class TicketState(TypedDict, total=False):
    question: str
    category: str
    evidence: str
    draft: str
    approved: bool
    final: str

def classify(state: TicketState):
    decision = structured(model, RouteDecision).invoke([HumanMessage(state["question"])])
    return {"category": decision.category}

def faq(state: TicketState):
    return {"evidence": search_policies.invoke({"query": state["question"]})}

def billing(state: TicketState):
    order_id = re.search(r"O\d{4}", state["question"])
    return {"evidence": get_order.invoke({"order_id": order_id.group(0)}) if order_id else "no order id in the question"}

def draft(state: TicketState):
    reply = model.invoke([SystemMessage("Draft a short customer reply from the evidence. Be factual.\n\nEvidence:\n" + state["evidence"]), HumanMessage(state["question"])])
    return {"draft": text_of(reply)}

def approve(state: TicketState):
    decision = interrupt({"draft": state["draft"], "question": "Send this reply to the customer?"})   # pauses here
    return {"approved": bool(decision)}

def send(state: TicketState):
    return {"final": state["draft"] if state["approved"] else "Reply withheld by reviewer."}

print("nodes defined:", [f.__name__ for f in (classify, faq, billing, draft, approve, send)])

### Step 2 — Wire the graph, compile with a checkpointer, run to the pause

In [ ]:
builder = StateGraph(TicketState)
for node in (classify, faq, billing, draft, approve, send):
    builder.add_node(node.__name__, node)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", lambda state: state["category"], {"faq": "faq", "billing": "billing"})
builder.add_edge("faq", "draft")
builder.add_edge("billing", "draft")
builder.add_edge("draft", "approve")
builder.add_edge("approve", "send")
builder.add_edge("send", END)
ticket_graph = builder.compile(checkpointer=InMemorySaver())

print(ticket_graph.get_graph().draw_mermaid())        # the same picture as a Mermaid diagram

run = {"configurable": {"thread_id": "wf-1"}}
paused = ticket_graph.invoke({"question": "I was charged twice for order O1002. What happens now?"}, run)
print("category :", paused["category"])
print("evidence :", paused["evidence"][:90])
print("paused at:", ticket_graph.get_state(run).next, "| asks:", paused["__interrupt__"][0].value["question"])

### Step 3 — Resume, and see durability

The reviewer approves; the graph continues from the `approve` node, not from the start.
`get_state_history()` lists every checkpoint: this is what makes crash recovery and
"time travel" debugging possible, and why long-running agents are built on persistence.

In [ ]:
finished = ticket_graph.invoke(Command(resume=True), run)
print("final    :", finished["final"][:140])

history = list(ticket_graph.get_state_history(run))
print("\ncheckpoints recorded:", len(history))
for snap in reversed(history):
    print("  next =", snap.next or ("END",), "| keys so far:", sorted(k for k in snap.values if k != "question"))

### Recap

- **Problem seen:** a fixed business process was being left to a free-form agent loop.
- **Layer added:** an explicit LangGraph: typed state, nodes, conditional edges, `interrupt()`, checkpoints and state history.
- **Evidence:** the ticket paused at approval, resumed from that exact node, and every step was recorded as a checkpoint.